In [ ]:
# 在Colab环境中安装必要的依赖包
!pip install --upgrade pip --quiet
!pip install torch torchvision torchaudio --quiet
!pip install scipy numpy PyWavelets --quiet
!pip install transformers --quiet
!pip install scikit-learn --quiet


import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import scipy.signal as signal
import pandas as pd
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.signal import butter, filtfilt
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# 设置设备
device = torch.device("cuda")
print("使用设备:", device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 171.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 25.4 MB/s eta 0:00:00
使用设备: cuda


In [ ]:
########################################

FIELD_STRUCTURES = {
    268: {
        'record_length': 268,
        'ts_count': 3,        # hour, minute, second
        'rssi_count': 4,      # 4个RSSI
        'mcs_count': 1,
        'gain_count': 4,
        'csi_count': 64*4,    # 256
        'csi_num_subcarriers': 64, # 64个子载波
        'csi_num_antennas': 4,
        'target_fs': 100      # 目标采样率 100
    },
    1000: {
        'record_length': 1000,
        'ts_count': 3,
        'rssi_count': 2,      # 2个RSSI
        'mcs_count': 1,
        'gain_count': 2,
        'csi_count': 248*4,   # 992
        'csi_num_subcarriers': 248,
        'csi_num_antennas': 4,
        'target_fs': 20       # 目标采样率 20
    },
    1008: {
        'record_length': 1008,
        'ts_count': 3,
        'rssi_count': 2,
        'mcs_count': 1,
        'gain_count': 2,
        'csi_count': 250*4,   # 1000
        'csi_num_subcarriers': 250,
        'csi_num_antennas': 4,
        'target_fs': 20
    },
    # 如还有其他结构，可再添加
}


def match_field_structure(file_name):
    base = os.path.basename(file_name)

    if "csi_268_" in base:
        return FIELD_STRUCTURES[268]
    elif "csi_1008_" in base:
        return FIELD_STRUCTURES[1008]
    elif "csi_1000_" in base:
        return FIELD_STRUCTURES[1000]
    else:
        # fallback
        print(f"[Warning] 未匹配到特定结构, 默认使用 csi_268 => file_name={base}")
        return FIELD_STRUCTURES["268"]



########################################
# (C) 函数: read_csi_txt
########################################



def read_csi_txt(file_path):
    """
    读取单个 txt 文件, 解析出 data_packets(list of dict).
    """
    base = os.path.basename(file_path)
    field_structure = match_field_structure(base)
    record_length = field_structure["record_length"]

    # 打开文件
    with open(file_path, 'r') as f:
        all_str = f.read().strip().split()  # 全部按空格拆分
    total_elems = len(all_str)
    total_records = total_elems // record_length
    extra = total_elems % record_length

    print(f"文件 {base} => record_length={record_length}, total_elems={total_elems}, total_records={total_records}, extra={extra}")
    if extra != 0:
        print(f"  [截断] 移除最后 {extra} 个元素")
        all_str = all_str[: total_records*record_length]

    data_records = []
    idx=0
    for i in range(total_records):
        start = i*record_length
        seg   = all_str[start:start+record_length]

        # 逐字段解析
        # 1) ts
        ts_count  = field_structure["ts_count"]
        ts_vals   = seg[0:ts_count]
        ts = [float(x) for x in ts_vals]

        # 2) rssi
        rssi_count= field_structure["rssi_count"]
        rssi_vals = seg[ts_count : ts_count + rssi_count]
        rssi = [float(x) for x in rssi_vals]

        # 3) mcs
        mcs_count = field_structure["mcs_count"]
        mcs_start = ts_count + rssi_count
        mcs_vals  = seg[mcs_start : mcs_start+mcs_count]
        mcs = float(mcs_vals[0]) if mcs_count>0 else 0.

        # 4) gain
        gain_count= field_structure["gain_count"]
        gain_start= mcs_start + mcs_count
        gain_vals = seg[gain_start : gain_start+gain_count]
        gain = [float(x) for x in gain_vals]

        # 5) csi
        csi_count = field_structure["csi_count"]
        csi_start = gain_start + gain_count
        csi_vals  = seg[csi_start : csi_start + csi_count]
        # 解析成复数
        csi_list=[]
        for val_str in csi_vals:
            val_str=val_str.replace('I','j').replace('i','j')
            try:
                val_comp=complex(val_str)
            except:
                val_comp=0+0j
            csi_list.append(val_comp)
        csi_array=np.array(csi_list,dtype=np.complex128)

        # reshape => (num_subcarriers, num_antennas)
        subc = field_structure["csi_num_subcarriers"]
        ant  = field_structure["csi_num_antennas"]
        csi_array=csi_array.reshape(subc,ant)

        # fs
        final_fs=field_structure["target_fs"]

        data_records.append({
            "ts": ts,           # [hour,minute,second]...
            "rssi": rssi,       # list of float
            "mcs":  mcs,
            "gain": gain,
            "csi":  csi_array,  # shape (subc, ant)
            "fs":   final_fs
        })

    return data_records


########################################
# (D) 函数: read_truth_txt
########################################

def read_truth_txt(file_path):
    with open(file_path, 'r') as f:
        lines = f.read().strip().split()
    labels = [int(x) for x in lines]
    return labels



########################################
# (E) 插值 / CSI缩放 / 多普勒谱函数
########################################

def dbinv(x_db):
    return 10**(x_db / 10.0)


def get_total_rss(rssi):
    """
    计算平均 RSSI 的功率（dB）。

    参数:
    - rssi: 包含 RSSI 值的列表

    返回:
    - avg_linear_rss_db: 平均 RSSI 的功率（dB）
    """
    linear_vals = [dbinv(val) for val in rssi]
    avg_linear_rss = np.mean(linear_vals)
    return 10 * np.log10(avg_linear_rss)



def interpolate_csi(df, target_time, num_subcarriers=30, num_antennas=4):
    """
    对 CSI 数据进行插值，统一采样率到100 Hz。

    参数:
    - df: 包含 CSI 数据的 DataFrame，必须包含 'total_seconds' 和 'data' 列
    - target_time: 均匀时间网格数组
    - num_subcarriers: 子载波数（默认为30）
    - num_antennas: 天线数（默认为4）

    返回:
    - csi_uniform: 插值后的 CSI 数据，形状为 (len(target_time), num_subcarriers, num_antennas)
    """
    T = len(target_time)
    csi_uniform = np.zeros((T, num_subcarriers, num_antennas), dtype=np.complex128)

    original_time = df['total_seconds'].values

    for sc in range(num_subcarriers):
        for ant in range(num_antennas):
            # 提取原始 CSI 数据
            csi_complex = np.array([rec['csi'][sc, ant] for rec in df['data']])
            # 提取实部和虚部
            csi_real = csi_complex.real
            csi_imag = csi_complex.imag

            # 创建插值函数
            interp_real = interp1d(original_time, csi_real, kind='linear', fill_value='extrapolate')
            interp_imag = interp1d(original_time, csi_imag, kind='linear', fill_value='extrapolate')

            # 插值到均匀时间网格
            csi_real_uniform = interp_real(target_time)
            csi_imag_uniform = interp_imag(target_time)

            # 重构复数 CSI 数据
            csi_uniform[:, sc, ant] = csi_real_uniform + 1j * csi_imag_uniform

    return csi_uniform


def interpolate_rssi(data, target_time, num_rssi=4):
    """
    对 RSSI 数据进行插值，统一采样率到100 Hz。

    参数:
    - data: 包含 CSI 数据的列表，每个元素包含 'rssi' 键
    - target_time: 均匀时间网格数组
    - num_rssi: 每条记录的 RSSI 数量（默认为4）

    返回:
    - rssi_uniform: 插值后的 RSSI 数据，形状为 (len(target_time), num_rssi)
    """
    T = len(target_time)
    rssi_uniform = np.zeros((T, num_rssi), dtype=np.float32)

    original_time = np.array([rec['ts'][0]*3600 + rec['ts'][1]*60 + rec['ts'][2] for rec in data])

    for r in range(num_rssi):
        # 提取第 r 个 RSSI 值，如果某条记录的 RSSI 不足 r+1 个，则使用最后一个 RSSI 值填充
        rssi_values = []
        for rec in data:
            if len(rec['rssi']) > r:
                rssi_values.append(rec['rssi'][r])
            else:
                rssi_values.append(rec['rssi'][-1])

        # 创建插值函数
        interp_func = interp1d(original_time, rssi_values, kind='linear', fill_value='extrapolate')

        # 插值到均匀时间网格
        rssi_uniform[:, r] = interp_func(target_time)

    return rssi_uniform

def get_scaled_csi_single(csi, rssi, noise_db=-92, Ntx=1, Nrx=3):
    """
    缩放 CSI 数据。

    参数:
    - csi: 原始 CSI 数据（64x4 矩阵）
    - rssi: RSSI 值列表
    - noise_db: 噪声功率（dB）
    - Ntx: 发送天线数
    - Nrx: 接收天线数

    返回:
    - scaled_csi: 缩放后的 CSI 数据
    """
    csi = csi.astype(np.complex128, copy=False)
    total_rss_db = get_total_rss(rssi)
    rssi_pwr = dbinv(total_rss_db)

    csi_sq = csi * np.conjugate(csi)
    csi_pwr = np.sum(csi_sq)
    scale = rssi_pwr / (csi_pwr / 30.0)

    thermal_noise_pwr = dbinv(noise_db)
    quant_error_pwr = scale * (Nrx * Ntx)
    total_noise_pwr = thermal_noise_pwr + quant_error_pwr

    scaled_csi = csi * np.sqrt(scale / total_noise_pwr)
    if Ntx == 2:
        scaled_csi *= np.sqrt(2)
    elif Ntx == 3:
        scaled_csi *= np.sqrt(dbinv(4.5))
    return scaled_csi


def design_bandpass_filter(lowcut, highcut, fs, order=5):
    """
    设计一个带通滤波器，并返回滤波器系数。

    参数:
    - lowcut: 带通滤波器的低截止频率（Hz）
    - highcut: 带通滤波器的高截止频率（Hz）
    - fs: 采样率（Hz）
    - order: 滤波器的阶数

    返回:
    - b, a: 滤波器系数
    """
    nyq = 0.5 * fs  # 奈奎斯特频率
    # 确保 highcut 不超过奈奎斯特频率的 99%
    highcut = min(highcut, 0.99 * nyq)
    # 确保 lowcut 大于 0，并且小于 highcut
    lowcut = max(lowcut, 0.01 * nyq)
    if lowcut >= highcut:
        raise ValueError(f"低截止频率 ({lowcut} Hz) 必须小于高截止频率 ({highcut} Hz)。")

    Wn = [lowcut / nyq, highcut / nyq]

    # 检查规范化截止频率是否在 (0,1) 之间
    if not (0 < Wn[0] < Wn[1] < 1):
        raise ValueError(f"规范化截止频率 Wn={Wn} 不在 (0,1) 之间。请检查 lowcut 和 highcut 的设置。")

    b, a = butter(order, Wn, btype='band')
    return b, a


def apply_bandpass_filter(data, lowcut, highcut, fs, order=5):
    """
    应用带通滤波器到数据。

    参数:
    - data: 待滤波的数据（1D NumPy 数组）
    - lowcut: 带通滤波器的低截止频率（Hz）
    - highcut: 带通滤波器的高截止频率（Hz）
    - fs: 采样率（Hz）
    - order: 滤波器的阶数

    返回:
    - y: 滤波后的数据
    """
    b, a = design_bandpass_filter(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data)
    return y


def get_doppler_spectrum_from_tensor(csi_tensor, timestamps,
                                     rx_cnt=1, rx_acnt=4,
                                     method='stft',
                                     fs=100.0,                # 固定目标采样率
                                     window_size_seconds=2.0,
                                     hop_size_seconds=1.5):
    """
    生成多普勒谱。

    参数:
    - csi_tensor: 缩放后的CSI数据，形状为 (T, 30, 4)
    - timestamps: 时间戳数组
    - rx_cnt: 接收天线数（默认1）
    - rx_acnt: 每个接收天线的子天线数（默认4）
    - method: 生成多普勒谱的方法（默认'stft'）
    - fs: 采样率（Hz）
    - window_size_seconds: 窗口大小（秒）
    - hop_size_seconds: 窗口跳跃大小（秒）

    返回:
    - doppler_spectrum: 多普勒谱，形状为 (rx_cnt, freq_bins, time_frames)
    - freq_bin: 频率轴
    - frame_timestamps: 每帧的时间戳
    """
    samp_rate = fs
    half_rate = samp_rate / 2

    # 设定带通滤波频率
    lowcut = 2.0    # 低端2Hz
    highcut = 40.0  # 高端40Hz

    # Clamp highcut
    if highcut > 0.99 * half_rate:
        print(f"[clamp] highcut={highcut}超出Nyquist={half_rate},自动缩小")
        highcut = 0.99 * half_rate
    if lowcut < 0.01:
        lowcut = 0.01
    if lowcut >= highcut:
        lowcut = 1.0
        highcut = 0.99 * half_rate

    print(f"[Info] 采样率 fs={samp_rate}Hz, lowcut={lowcut}Hz, highcut={highcut}Hz")

    # 设计带通滤波器
    try:
        b_band, a_band = design_bandpass_filter(lowcut, highcut, samp_rate, order=5)
    except ValueError as e:
        print(f"[滤波器设计错误]: {e}")
        return np.zeros((rx_cnt, 0, 0), dtype=np.float32), None, None

    doppler_spectrum_list = []
    freq_bin = None
    frame_timestamps = []

    for ii in range(rx_cnt):
        start_antenna = ii * rx_acnt
        end_antenna   = start_antenna + rx_acnt
        # 选前30子载波
        csi_data = csi_tensor[:, :30, start_antenna:end_antenna].reshape(-1, 30 * rx_acnt)

        # 选择最佳天线对
        csi_mean = np.mean(np.abs(csi_data), axis=0)
        csi_var  = np.sqrt(np.var(np.abs(csi_data), axis=0) + 1e-9)
        csi_mean_var_ratio = csi_mean / csi_var
        try:
            csi_mean_var_ratio_2d = csi_mean_var_ratio.reshape((30, rx_acnt), order='F')
        except ValueError:
            print(f"[reshape失败], skip rx={ii}")
            continue
        idx = np.argmax(np.mean(csi_mean_var_ratio_2d, axis=0))
        start_col = idx * 30
        end_col   = (idx + 1) * 30
        csi_data_ref = np.tile(csi_data[:, start_col:end_col], (1, rx_acnt))

        # 幅度调整
        csi_data_adj = np.zeros_like(csi_data, dtype=np.complex128)
        csi_data_ref_adj = np.zeros_like(csi_data_ref, dtype=np.complex128)
        alpha_sum = 0
        for jj in range(30 * rx_acnt):
            amp = np.abs(csi_data[:, jj])
            amp_nonzero = amp[amp > 0]
            alpha = np.min(amp_nonzero) if len(amp_nonzero) > 0 else 1e-6
            alpha_sum += alpha
            csi_data_adj[:, jj] = np.maximum(amp - alpha, 0) * np.exp(1j * np.angle(csi_data[:, jj]))
        beta = 1000 * alpha_sum / (30 * rx_acnt)
        for jj in range(30 * rx_acnt):
            amp_ref = np.abs(csi_data_ref[:, jj])
            csi_data_ref_adj[:, jj] = (amp_ref + beta) * np.exp(1j * np.angle(csi_data_ref[:, jj]))

        # conj_mult
        conj_mult = csi_data_adj * np.conjugate(csi_data_ref_adj)
        # 去除 idx
        conj_mult = np.concatenate([conj_mult[:, :start_col], conj_mult[:, end_col:]], axis=1)

        # 带通滤波
        for jj in range(conj_mult.shape[1]):
            try:
                conj_mult[:, jj] = filtfilt(b_band, a_band, conj_mult[:, jj])
            except Exception as ex:
                print(f"[Filter失败@列{jj}]: {ex}")
                conj_mult[:, jj] = 0 + 0j

        # PCA(只保留第1主成分)
        try:
            U, S, Vh = np.linalg.svd(conj_mult, full_matrices=False)
            # conj_mult_pca => shape (T,)
            conj_mult_pca = conj_mult @ Vh.conjugate().T[:, 0]
        except np.linalg.LinAlgError as ex:
            print(f"[PCA失败]: {ex}")
            continue

        # STFT
        if method.lower() == 'stft':
            window_size_samples = int(round(window_size_seconds * samp_rate))  # 2.0 * 100 = 200
            hop_size_samples    = int(round(hop_size_seconds * samp_rate))     # 1.5 * 100 = 150

            print(f"nperseg: {window_size_samples}")
            nfft = max(window_size_samples, 512)  # 设置 nfft 至至少 nperseg，且通常选择 2 的幂次方
            print(f"nfft: {nfft}")
            window = signal.windows.gaussian(window_size_samples, std=window_size_samples / 6)
            f, t, Zxx = signal.stft(
                conj_mult_pca,
                fs=samp_rate,
                window=window,
                nperseg=window_size_samples,
                noverlap=window_size_samples - hop_size_samples,
                nfft=nfft,  # 明确设置 nfft
                boundary=None
            )
            print(f"STFT result: f.shape={f.shape}, t.shape={t.shape}, Zxx.shape={Zxx.shape}")
            freq_time_prof_allfreq = Zxx
            # 选频
            freq_lpf_sel = (f <= highcut)  # 仅保留 0 到 highcut 之间的频率
            freq_time_prof = freq_time_prof_allfreq[freq_lpf_sel, :]
            freq_bin = f[freq_lpf_sel]
            # 幅度 & 每帧归一化
            freq_time_prof = np.abs(freq_time_prof)
            sum_val = np.sum(freq_time_prof, axis=0, keepdims=True) + 1e-9
            freq_time_prof = freq_time_prof / sum_val
            frame_timestamps = t
        else:
            # cwt 略示意
            freq_bin = None
            frame_timestamps = None
            freq_time_prof = np.empty((0, 0), dtype=np.float32)

        doppler_spectrum_list.append(freq_time_prof)

    if len(doppler_spectrum_list) == 0:
        return np.zeros((rx_cnt, 0, 0), dtype=np.float32), None, None
    doppler_spectrum = np.stack(doppler_spectrum_list, axis=0).astype(np.float32)
    print(f"Doppler Spectrum shape: {doppler_spectrum.shape}")  # (rx_cnt, freq_dim, time_frames)

    return doppler_spectrum, freq_bin, frame_timestamps


########################################
# (F) load_data_from_folder
########################################

import os
import numpy as np
import pandas as pd
import random
#######################
# 假设以下函数你已有：
# read_csi_txt(file_path) -> list of dict
#    dict 每项: {
#       "ts": [hr, min, sec], # 仅示例
#       "rssi": [ ... ],
#       "csi": shape=(?) (若 268 => (64,4); 1008=> (250,4) ...),
#       ...
#    }
#
# read_truth_txt(file_path) -> list of int
# interpolate_csi(df_temp, uniform_time, num_subcarriers=30, num_antennas=4) -> (T, 30,4)
# interpolate_rssi(data, uniform_time, num_rssi=4) -> (T,4)
# get_scaled_csi_single(...)-> shape=(30,4)
# get_doppler_spectrum_from_tensor(...) -> shape=(1, freq_dim, frames), ...
#######################


# 数据加载函数
def load_data_from_folder(folder_path):
    """
    读取某文件夹下的 CSI 文件，并生成多普勒谱 (幅度 + 相位)。
    每个谱图的形状为 (2, freq_dim_i, time_frames)，freq_dim_i 可能不同。
    同时保存每条窗口对应的 RSSI 和 label。

    返回:
        all_spectrograms: list of np.array, each shape=(2, freq_dim_i, time_frames)
        all_labels: np.array of shape=(N,)
        all_rssi_stats: list of np.array, each shape=(rssi_dim_i,)
    """
    all_data_records = []
    # 遍历文件夹中的所有 txt 文件，排除包含 '_truth' 的文件
    for file_name in sorted(os.listdir(folder_path)):
        if file_name.endswith(".txt") and "_truth" not in file_name:
            file_path = os.path.join(folder_path, file_name)
            try:
                data = read_csi_txt(file_path)   # 你自己的函数
                if data:
                    all_data_records.append({
                        "file_name": file_name,
                        "data": data
                    })
                else:
                    print(f"文件 {file_name} 中没有有效的记录。")
            except Exception as e:
                print(f"文件 {file_name} 解析失败: {e}")






    # 确保有数据可选
    if all_data_records:
        random_record = random.choice(all_data_records)  # 从列表中随机选取一个文件
        file_name = random_record["file_name"]
        data = random_record["data"]

        print(f"\n随机选取的数据来自文件: {file_name}")

        # 确保数据列表有足够的条目
        if len(data) >= 2:
            first_packet = data[0]  # 第一条数据
            second_packet = data[1]  # 第二条数据
            last_packet = data[-1]  # 最后一条数据

            def print_packet(packet, index):
                print(f"\n=== 第 {index} 条 CSI 数据包 ===")
                print(f"Timestamp (ts): {packet['ts']}")
                print(f"RSSI: {packet['rssi']}")
                print(f"MCS: {packet['mcs']}")
                print(f"Gain: {packet['gain']}")
                print("CSI (full data):")
                print(packet['csi'])  # 完整展示CSI数据
                #print(f"Sampling Rate (fs): {packet['fs']}")

            print_packet(first_packet, "1")
            print_packet(second_packet, "2")
            print_packet(last_packet, "最后")
        else:
            print(f"文件 {file_name} 记录数不足 2 条，无法打印完整示例。")
    else:
        print("没有数据文件可选。")





    print(f"[load_data_from_folder] 读取 {len(all_data_records)} 个文件于 {folder_path}")

    all_spectrograms = []
    all_labels       = []
    all_rssi_stats   = []

    for file_entry in all_data_records:
        file_name = file_entry['file_name']
        data      = file_entry['data']

        prefix = file_name[:-4]
        truth_file = prefix + "_truth.txt"
        truth_path = os.path.join(folder_path, truth_file)

        print(f"\n处理文件 {file_name}: ")
        if len(data) == 0:
            print("  数据为空，跳过。")
            continue

        if os.path.exists(truth_path):
            truth_labels = read_truth_txt(truth_path)
            num_labels   = len(truth_labels)
            print(f"  对应 truth 数目: {num_labels}")
        else:
            print(f"  无 {truth_file}, 跳过")
            continue

        # 1) 提取时间戳 & 采样率
        timestamps = np.array([
            rec['ts'][0]*3600 + rec['ts'][1]*60 + rec['ts'][2] for rec in data
        ])
        start_time = timestamps.min()
        end_time   = timestamps.max()

        final_fs= data[0]['fs']  # 取第一个即可
        target_dt = 1.0 / final_fs
        uniform_time = np.arange(start_time, end_time, target_dt)
        num_rssi = len(data[0]['rssi'])
        print(f"  目标采样率: {final_fs} Hz, 均匀时间长度: {len(uniform_time)}, num_rssi ={num_rssi}")

        # 2) 插值
        df_temp = pd.DataFrame({'total_seconds': timestamps, 'data': data})
        csi_uniform  = interpolate_csi(df_temp, uniform_time, num_subcarriers=30, num_antennas=4)
        rssi_uniform = interpolate_rssi(data, uniform_time, num_rssi)
        print("  插值后 csi:", csi_uniform.shape, "rssi:", rssi_uniform.shape)

        # 3) 缩放
        scaled_tensor = np.zeros_like(csi_uniform, dtype=np.complex128)
        for t in range(len(uniform_time)):
            csi_val  = csi_uniform[t]
            rssi_val = rssi_uniform[t]
            scaled_csi = get_scaled_csi_single(
                csi_val, rssi_val, noise_db=-92, Ntx=1, Nrx=4
            )
            # 这里假设已统一截取前30个子载波
            scaled_tensor[t] = scaled_csi[:30, :4]
        print(f"  scaled_tensor shape={scaled_tensor.shape}")

        # 4) 生成多普勒谱 (复数)
        doppler_complex, freq_bin, frame_ts = get_doppler_spectrum_from_tensor(
            scaled_tensor, timestamps,
            rx_cnt=1, rx_acnt=4,
            method='stft',
            fs=final_fs,
            window_size_seconds=2.0,
            hop_size_seconds=0.5
        )
        if doppler_complex.shape[2] == 0:
            print("  多普勒谱为空，跳过。")
            continue

        freq_dim = doppler_complex.shape[1]
        t_frames = doppler_complex.shape[2]
        print(f"  doppler谱 shape={doppler_complex.shape} (freq_dim={freq_dim}, time_frames={t_frames})")

        # 幅度 + 相位
        amp_spectrum   = np.abs(doppler_complex)    # shape: (1, freq_dim, time_frames)
        phase_spectrum = np.angle(doppler_complex)  # shape: (1, freq_dim, time_frames)

        # 5) 按 2s窗口 (4帧) 切分
        window_size_seconds = 2.0
        hop_size_seconds    = 1.5

        frames_per_window = int(window_size_seconds / 0.5)  #
        number_of_windows = (t_frames - 1)// frames_per_window + 1

        # 让 number_of_windows 跟 label 数量对齐
        if number_of_windows > num_labels:
            number_of_windows = num_labels
        elif number_of_windows < num_labels:
            # 说明 label 多余 => 截断 label
            truth_labels = truth_labels[:number_of_windows]

        for i in range(number_of_windows):
            start_f = i * frames_per_window
            end_f   = start_f + frames_per_window
            if end_f > t_frames:
                break

            sub_amp   = amp_spectrum[0, :, start_f:end_f]   # => (freq_dim_i, 4)
            sub_phase = phase_spectrum[0, :, start_f:end_f] # => (freq_dim_i, 4)
            multi_2d  = np.stack([sub_amp, sub_phase], axis=0)  # => (2, freq_dim_i, 4)

            all_spectrograms.append(multi_2d)
            all_labels.append(truth_labels[i])

            # RSSI
            mid_f = (start_f + end_f)//2
            mid_time = frame_ts[mid_f]
            idx_time = np.argmin(np.abs(uniform_time - mid_time))
            rssi4 = rssi_uniform[idx_time]  # => shape=(4,)
            all_rssi_stats.append(rssi4.copy())

    print(f"\n[load_data_from_folder] 共收集 {len(all_spectrograms)} 个谱图, {len(all_labels)} 个标签.")

    # 不做“硬性” freq_dim 统一 => 返回可变
    # all_spectrograms: list of (2, freq_dim_i, 4), freq_dim_i 各不相同
    # all_rssi_stats: list of np.array, each shape=(rssi_dim_i,)
    # all_labels:     np.array of shape=(N,)
    return all_spectrograms, np.array(all_labels, dtype=np.int64), all_rssi_stats







ModuleNotFoundError: No module named 'pandas'

In [ ]:

class DopplerRSSIDataset(Dataset):
    def __init__(self, all_spectrograms, all_labels, all_rssi_stats):
        """
        all_spectrograms: list of np.array, each shape=(2, freq_dim_i, 4)
        all_labels: np.array of shape=(N,)
        all_rssi_stats: list of np.array, each shape=(rssi_dim_i,)
        """
        self.all_spectrograms = all_spectrograms
        self.all_labels       = all_labels
        self.all_rssi_stats   = all_rssi_stats
        assert len(self.all_spectrograms) == len(self.all_labels) == len(self.all_rssi_stats)

    def __len__(self):
        return len(self.all_spectrograms)

    def __getitem__(self, idx):
        doppler_2d = self.all_spectrograms[idx]  # shape=(2, freq_dim_i, 4), numpy
        label      = self.all_labels[idx]
        rssi_arr   = self.all_rssi_stats[idx]    # shape=(rssi_dim_i,), list

        # 转换为 torch tensor
        doppler_tensor = torch.from_numpy(doppler_2d).float()  # (2, freq_dim_i, 4)
        rssi_tensor    = torch.from_numpy(rssi_arr).float()     # (rssi_dim_i,)

        return doppler_tensor, rssi_tensor, label


def doppler_collate_fn(batch):
    """
    batch: list of tuples (doppler_tensor, rssi_tensor, label)
        doppler_tensor: (2, freq_dim_i, 4)
        rssi_tensor: (rssi_dim_i,)
        label: int
    返回:
        doppler_list: list of torch.Tensor, each shape=(2, freq_dim_i, 4)
        rssi_list: list of torch.Tensor, each shape=(rssi_dim_i,)
        label_tensor: torch.Tensor of shape=(B,)
    """
    doppler_list = []
    rssi_list    = []
    label_list   = []
    for (dopp_t, rssi_t, lab) in batch:
        doppler_list.append(dopp_t)      # (2, freq_dim_i, 4)
        rssi_list.append(rssi_t)        # (rssi_dim_i,)
        label_list.append(lab)           # int
    label_tensor = torch.tensor(label_list, dtype=torch.long)
    return doppler_list, rssi_list, label_tensor


In [ ]:

class RSSIProjector(nn.Module):
    """
    演示用的 RSSI 投影层: 把可变维度的 rssi(<=max_rssi_dim) 填充/截断到 max_rssi_dim，
    再用一个全连接投影到固定 rssi_proj_dim 维。
    """
    def __init__(self, max_rssi_dim=4, proj_dim=8):
        super().__init__()
        self.max_rssi_dim = max_rssi_dim
        self.proj = nn.Linear(max_rssi_dim, proj_dim)

    def forward(self, rssi_tensor):
        # rssi_tensor: shape=(rssi_dim_i,)  可能 < max_rssi_dim or =max_rssi_dim
        rssi_dim = rssi_tensor.shape[0]
        # 1) 若 rssi_dim < max_rssi_dim => padding
        #    若 rssi_dim > max_rssi_dim => 截断
        pad_rssi = torch.zeros(self.max_rssi_dim, device=rssi_tensor.device, dtype=rssi_tensor.dtype)
        if rssi_dim >= self.max_rssi_dim:
            pad_rssi[:] = rssi_tensor[:self.max_rssi_dim]
        else:
            pad_rssi[:rssi_dim] = rssi_tensor

        # 2) 全连接投影 => (proj_dim,)
        out = self.proj(pad_rssi)  # => shape=(proj_dim,)
        return out

class DopplerRSSI_FusionModel(nn.Module):
    def __init__(self,
                 task_type='classification',
                 doppler_in_channels=2,  # (2=幅度+相位)
                 out_freq_dim=128,       # 自适应池化后 freq_dim
                 time_dim=4,             # time_dim 固定为4
                 doppler_hidden_size=64, # LSTM hidden size for Doppler
                 max_rssi_dim=4,         # 最大 RSSI 维度
                 rssi_proj_dim=8,        # 投影后的 RSSI 维度
                 rssi_hidden_size=32,    # LSTM hidden size for RSSI
                 num_classes=4):
        super(DopplerRSSI_FusionModel, self).__init__()
        self.task_type = task_type
        self.out_freq_dim = out_freq_dim
        self.time_dim     = time_dim
        self.doppler_in_channels = doppler_in_channels

        # ========== 1) Doppler 分支: CNN + Adaptive Pooling + MaxPool + LSTM ==========
        self.conv1 = nn.Conv2d(doppler_in_channels, 8, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(8)
        self.relu  = nn.ReLU()
        self.adaptive_pool = nn.AdaptiveAvgPool2d((out_freq_dim, time_dim))
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(16)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)

        # 可以在 CNN 输出后/前插入 dropout
        self.dropout_cnn = nn.Dropout(p=0.3)

        # LSTM for Doppler
        # 如果仅 num_layers=1，LSTM内置的dropout并不会生效
        # 下面演示在 LSTM 输出后手动加 dropout
        self.doppler_lstm = nn.LSTM(
            input_size=16 * (out_freq_dim // 2),
            hidden_size=doppler_hidden_size,
            num_layers=1,
            batch_first=True
            # dropout=0.3, # 只有 num_layers>1 时生效
        )
        # 手动在 LSTM 输出后添加 dropout
        self.dropout_lstm = nn.Dropout(p=0.3)

        # ========== 2) RSSI 分支: 投影 + LSTM ==========
        self.rssi_proj = RSSIProjector(max_rssi_dim=max_rssi_dim, proj_dim=rssi_proj_dim)
        self.rssi_lstm = nn.LSTM(
            input_size=rssi_proj_dim,
            hidden_size=rssi_hidden_size,
            num_layers=1,
            batch_first=True
        )
        self.dropout_rssi = nn.Dropout(p=0.3)  # 手动加

        # ========== 3) 融合全连接层 ==========
        fusion_dim = doppler_hidden_size + rssi_hidden_size

        # 根据任务类型设置输出层
        if self.task_type == 'classification':
            self.fc_fusion = nn.Linear(fusion_dim, num_classes)
        elif self.task_type == 'regression':
            self.fc_fusion = nn.Linear(fusion_dim, 1)
        else:
            raise ValueError(f"Unsupported task type: {task_type}")



    def forward(self, doppler_list, rssi_list):
        """
        doppler_list: list of B tensors, each shape=(2, freq_dim_i, 4)
        rssi_list:   list of B tensors, each shape=(rssi_dim_i,)

        返回: logits: (B, num_classes)
        """
        B = len(doppler_list)

        # 1) 处理 RSSI 分支
        rssi_proj_batch = []
        for i in range(B):
            rssi = rssi_list[i].to(next(self.parameters()).device)
            # 先投影 => (proj_dim,)
            r_proj = self.rssi_proj(rssi)      # => shape=(proj_dim,)
            # => (1, seq_len=1, proj_dim)
            r_proj = r_proj.unsqueeze(0).unsqueeze(0)
            out_r, _ = self.rssi_lstm(r_proj)  # => (1,1,rssi_hidden_size)
            hidden_rssi = out_r[:, -1, :]      # =>(1,rssi_hidden_size)
            hidden_rssi = self.dropout_rssi(hidden_rssi)  # dropout
            rssi_proj_batch.append(hidden_rssi)

        # 拼接成 (B, rssi_hidden_size)
        rssi_batch = torch.cat(rssi_proj_batch, dim=0)

        # 2) 处理 Doppler 分支
        doppler_hidden_batch = []
        for i in range(B):
            x = doppler_list[i].to(next(self.parameters()).device)  # shape=(2,freq_dim_i, 4)
            x = x.unsqueeze(0)  # =>(1,2,freq_dim_i,4)
            # CNN
            x = self.conv1(x)
            x = self.bn1(x)
            x = self.relu(x)

            x = self.adaptive_pool(x)    # =>(1,8,out_freq_dim, time_dim)
            x = self.conv2(x)
            x = self.bn2(x)
            x = self.relu(x)
            x = self.pool(x)             # =>(1,16,out_freq_dim//2,time_dim//2)

            x = self.dropout_cnn(x)      # CNN后再 dropout

            # Reshape => LSTM
            # x.shape=(1,16,out_freq_dim//2, time_dim//2)
            _, C, Fh, Tw = x.shape
            x = x.view(1, C*Fh, Tw)      # =>(1, 16*(out_freq_dim//2), time_dim//2)
            x = x.permute(0,2,1)         # =>(1, time_dim//2, 16*(out_freq_dim//2))

            out_d, _= self.doppler_lstm(x)  # =>(1, time_dim//2, doppler_hidden_size)
            hidden_doppler= out_d[:, -1, :] # =>(1, doppler_hidden_size)
            hidden_doppler= self.dropout_lstm(hidden_doppler)
            doppler_hidden_batch.append(hidden_doppler)

        doppler_batch = torch.cat(doppler_hidden_batch, dim=0)  # =>(B, doppler_hidden_size)

        # 3) 融合
        fusion = torch.cat([doppler_batch, rssi_batch], dim=1)  # =>(B, doppler_hidden_size + rssi_hidden_size)
        logits= self.fc_fusion(fusion)                          # =>(B, num_classes)

        # 根据任务类型选择输出方式
        if self.task_type == 'classification':
            return logits
        elif self.task_type == 'regression':
            return logits.squeeze()  # 对于回归任务，去除维度



# =============== 示例: 优化器+Weight Decay ===============
# 你可以这样写:
# model = DopplerRSSI_FusionModel(...)
# optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# dropout+p, weight_decay=..., 都可以在训练中当作正则化手段


In [ ]:
def train(model, device, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for doppler_list, rssi_list, labels in train_loader:
        # Move labels to device
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(doppler_list, rssi_list)  # (B, num_classes)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    epoch_loss = running_loss / running_total
    epoch_acc  = running_correct / running_total

    return epoch_loss, epoch_acc

def evaluate(model, device, val_loader, criterion):
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for doppler_list, rssi_list, labels in val_loader:
            labels = labels.to(device)

            outputs = model(doppler_list, rssi_list)  # (B, num_classes)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    epoch_loss = val_loss / val_total
    epoch_acc  = val_correct / val_total

    return epoch_loss, epoch_acc


In [ ]:
def get_data_loader_from_folder(folder_path, task_type):
    """
    根据任务类型（感知任务）和数据集结构从文件夹加载数据。

    参数：
    - folder_path: 数据所在文件夹路径
    - task_type: 感知任务类型（如分类、检测、定位等）

    返回：
    - train_loader: 训练数据加载器
    - val_loader: 验证数据加载器
    """
    # 加载数据
    all_spectrograms, all_labels, all_rssi_stats = load_data_from_folder(folder_path)

    # 处理不同的感知任务
    if task_type == 'classification':
        train_specs, val_specs, train_labels, val_labels, train_rssi, val_rssi = train_test_split(
            all_spectrograms, all_labels, all_rssi_stats, test_size=0.3, random_state=42)
    elif task_type == 'detection':
        # 针对检测任务的特定数据处理
        # 例如，做更复杂的标签处理，或增强数据等
        pass
    else:
        raise ValueError(f"Unknown task type: {task_type}")

    # 准备数据集
    train_dataset = DopplerRSSIDataset(train_specs, train_labels, train_rssi)
    val_dataset = DopplerRSSIDataset(val_specs, val_labels, val_rssi)

    # 数据加载器
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=doppler_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=doppler_collate_fn)

    return train_loader, val_loader


In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

def train_and_evaluate(task_type, train_folder):
    """
    训练和评估模型。

    参数:
    - task_type: 任务类型（'classification', 'regression'等）
    - train_folder: 训练数据文件夹路径
    - val_folder: 验证数据文件夹路径
    """
    # 加载数据
    train_loader, val_loader = get_data_loader_from_folder(train_folder, task_type)

    # 初始化模型
    model = DopplerRSSI_FusionModel(task_type=task_type, doppler_in_channels=2, out_freq_dim=128,
                                    time_dim=4, doppler_hidden_size=64, max_rssi_dim=4, rssi_proj_dim=8,
                                    rssi_hidden_size=32, num_classes=4).to(device)

    # 定义损失函数
    if task_type == 'classification':
        criterion = nn.CrossEntropyLoss()
    elif task_type == 'regression':
        criterion = nn.MSELoss()

    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-3)

    # 定义学习率调度器
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, verbose=True)

    # 定义早停参数
    patience = 50
    best_val_acc = 0.5
    epochs_no_improve = 0
    best_model_path = 'best_model.pth'

    # 训练循环
    num_epochs = 300
    for epoch in range(1, num_epochs + 1):
        # 训练
        train_loss, train_acc = train(model, device, train_loader , criterion, optimizer)

        # 验证
        val_loss, val_acc = evaluate(model, device, val_loader, criterion)

        # 更新学习率调度器
        scheduler.step(val_acc)

        # 记录最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> 新的最佳验证准确率: {best_val_acc:.4f}, 保存模型。")
        else:
            epochs_no_improve += 1

        # 检查早停条件
        if epochs_no_improve >= patience:

            print(f"早停触发在第 {epoch} 个 epoch。")
            break

        # 打印日志
        print(f"Epoch {epoch}/{num_epochs} | "
              f"Train Loss={train_loss:.4f}, Acc={train_acc:.4f} || "
              f"Val Loss={val_loss:.4f}, Acc={val_acc:.4f}")

    print("训练完成。")

# 调用训练和评估函数
train_folder = "/content/drive/MyDrive/hw-office"

task_type = 'classification'  # 或 'regression', 'detection'

train_and_evaluate(task_type, train_folder)


文件 csi_268_r0_2023_10_16_1.txt => record_length=268, total_elems=5717512, total_records=21334, extra=0
文件 csi_268_r0_2023_10_16_2.txt => record_length=268, total_elems=62983, total_records=235, extra=3
  [截断] 移除最后 3 个元素
文件 csi_268_r0_2023_10_16_2.txt 解析失败: could not convert string to float: '.1895+0.78125i'

随机选取的数据来自文件: csi_268_r0_2023_10_16_1.txt

=== 第 1 条 CSI 数据包 ===
Timestamp (ts): [0.0, 0.0, 0.0]
RSSI: [-28.0, -24.0, -22.0, -7.0]
MCS: 8.0
Gain: [9.0, 5.0, 2.0, -12.0]
CSI (full data):
[[ 1.5259e-05+1.5259e-05j  1.5259e-05+1.5259e-05j  1.5259e-05+1.5259e-05j
   1.5259e-05+1.5259e-05j]
 [-6.5430e-01-7.5781e-01j  3.5156e-02+1.1133e+00j  1.0371e+00-5.9766e-01j
  -1.0527e+00-6.4258e-01j]
 [-5.4492e-01+1.1211e+00j  1.1445e+00-5.3906e-01j -6.6016e-01-1.0742e+00j
  -1.2461e+00-1.4648e-01j]
 [-3.3203e-02+1.2715e+00j -5.6445e-01-1.1348e+00j -7.5391e-01+1.0215e+00j
   7.5195e-01+1.0195e+00j]
 [-4.6094e-01-1.2129e+00j -8.4570e-01+9.8242e-01j  1.2832e+00+2.9688e-01j
  -1.1309e+00-6.4062e-01j]


In [ ]:

# 测试阶段
test_folder  = "/content/drive/MyDrive/hw-office"   # 测试文件夹
test_specs, test_labels, test_rssi = load_data_from_folder(test_folder)
if len(test_specs) == 0:
    raise ValueError("测试数据为空，请检查测试文件夹。")

test_dataset = DopplerRSSIDataset(test_specs, test_labels, test_rssi)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=doppler_collate_fn)

# 确保加载最佳模型
best_model_path = 'best_model.pth'
model.load_state_dict(torch.load(best_model_path))  # 加载训练过程中保存的最佳模型
model.to(device)
model.eval()

# 测试阶段损失函数
if task_type == 'classification':
    criterion = nn.CrossEntropyLoss()
elif task_type == 'regression':
    criterion = nn.MSELoss()
else:
    raise ValueError(f"Unsupported task type: {task_type}")

# 测试评估
test_loss, test_acc = evaluate(model, device, test_loader, criterion)
print(f"[测试集结果] Loss={test_loss:.4f}, Acc={test_acc:.4f}")


文件 csi_268_r0_2023_10_16_1.txt => record_length=268, total_elems=5717512, total_records=21334, extra=0

随机选取的数据来自文件: csi_268_r0_2023_10_16_1.txt

=== 随机选取的 CSI 数据包 ===
Timestamp (ts): [0.0, 0.0, 24.6695]
RSSI: [-28.0, -24.0, -22.0, -7.0]
MCS: 8.0
Gain: [9.0, 5.0, 2.0, -12.0]
CSI (first few values): [[ 1.5259e-05+1.5259e-05j  1.5259e-05+1.5259e-05j  1.5259e-05+1.5259e-05j
   1.5259e-05+1.5259e-05j]
 [-1.0215e+00+8.5938e-02j  1.7773e-01+1.1484e+00j  1.2168e+00+2.2461e-01j
   1.2227e+00-3.1836e-01j]
 [-9.3750e-02+1.2891e+00j  1.2012e+00+5.2539e-01j -3.9062e-02-1.2910e+00j
  -9.9219e-01-8.1641e-01j]
 [-7.8320e-01+1.0332e+00j -1.0977e+00-7.5977e-01j  3.0664e-01+1.2754e+00j
   1.2930e+00+1.5430e-01j]
 [ 5.6250e-01-1.2129e+00j -1.2285e+00-5.1562e-01j  1.0195e+00+9.2773e-01j
   1.1680e+00-6.2695e-01j]]
Sampling Rate (fs): 100
[load_data_from_folder] 读取 1 个文件于 /content/drive/MyDrive/hw-office

处理文件 csi_268_r0_2023_10_16_1.txt: 
  对应 truth 数目: 151
  目标采样率: 100 Hz, 均匀时间长度: 30100, num_rssi =4
  插值

NameError: name 'model' is not defined